# Bước 6 — Error Analysis

Chạy lại `search()` (từ `search_core.py`) với 4 cấu hình ablation trên toàn bộ 30 câu trong `data/eval_set.jsonl`, so kết quả trả về với `relevant_ids` để tìm chính xác câu nào sai ở cấu hình nào — làm cơ sở gom nhóm nguyên nhân lỗi ở mục 12 của `BAO_CAO_TONG_HOP.md`.

Yêu cầu trước: đã chạy xong Bước 1-5 (có đủ `data/catalog.jsonl`, `dense.index`, `bm25.pkl`, `eval_set.jsonl`).

In [1]:
import json, sys
from pathlib import Path

# Notebook nằm ở gốc project, cùng cấp với search_core.py + data/
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from search_core import search

CATALOG_PATH = PROJECT_ROOT / "data" / "catalog.jsonl"
EVAL_SET_PATH = PROJECT_ROOT / "data" / "eval_set.jsonl"

catalog = {}
with open(CATALOG_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        catalog[row["id"]] = row

with open(EVAL_SET_PATH, encoding="utf-8") as f:
    eval_set = [json.loads(line) for line in f]

print(f"Đã load {len(catalog)} sản phẩm, {len(eval_set)} câu eval")

d:\Assignments\RAG\a1_multimodal_search\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã load 2000 sản phẩm, 30 câu eval


In [2]:
CONFIGS = [
    {"name": "bm25 only", "components": ["bm25"]},
    {"name": "dense only", "components": ["dense"]},
    {"name": "hybrid (bm25+dense)", "components": ["bm25", "dense"]},
    {"name": "hybrid + rerank", "components": ["bm25", "dense"], "use_rerank": True},
]
K = 10

misses = {}  # tên cấu hình -> list các câu eval bị MISS (không tìm thấy id đúng trong top-K)

for cfg in CONFIGS:
    cfg_misses = []
    for item in eval_set:
        results = search(
            item["query"], item["query_type"], k=K,
            components=cfg["components"], use_rerank=cfg.get("use_rerank", False),
        )
        retrieved_ids = [r["id"] for r in results]
        relevant = set(item["relevant_ids"])
        if not (set(retrieved_ids) & relevant):
            cfg_misses.append(item)
    misses[cfg["name"]] = cfg_misses
    print(f"{cfg['name']:<24} miss {len(cfg_misses)}/{len(eval_set)}")

bm25 only                miss 5/30
dense only               miss 6/30
hybrid (bm25+dense)      miss 2/30
hybrid + rerank          miss 3/30


In [4]:
# In chi tiết từng câu MISS: caption đúng là gì + top-3 kết quả hệ thống trả về
# thay vào -- đây là dữ liệu thô để tự đọc, gom nhóm nguyên nhân (không thể
# code sẵn phần "tự đọc và suy luận" này, xem mục 12 BAO_CAO_TONG_HOP.md).
for cfg in CONFIGS:
    cfg_name = cfg["name"]
    print(f"\n===== {cfg_name} =====")
    for item in misses[cfg_name]:
        rel_id = item["relevant_ids"][0]
        rel_row = catalog.get(rel_id, {})
        results = search(
            item["query"], item["query_type"], k=3,
            components=cfg["components"], use_rerank=cfg.get("use_rerank", False),
        )
        print(f"MISS  query='{item['query']}'")
        print(f"      expected id={rel_id} | brand={rel_row.get('brand')} | cat={rel_row.get('category')}")
        print(f"      expected caption: {rel_row.get('caption')}")
        print("      top-3 returned:")
        for r in results:
            row = catalog.get(r["id"], {})
            print(f"        id={r['id']} | brand={row.get('brand')} | caption={row.get('caption')}")


===== bm25 only =====
MISS  query='rug for stairs so people don't slip'
      expected id=241 | brand=VHC Brands | cat=Home & Garden > Household Supplies > Stair Treads
      expected caption: Cobblestone Jute Stair Tread - Rectangle
      top-3 returned:
        id=1839 | brand=HomeRoots | caption=9' X 13' Blue Oriental Stain Resistant Indoor Outdoor Area Rug
        id=1932 | brand=The Oxford Co | caption=Regular Punch Needles by The Oxford Co
        id=776 | brand=Stairs-Nosing | caption=19.5 x 20mm Anodised Aluminium Stair Edge Nosing with Anti Slip Design
MISS  query='inflatable towable tube for boating'
      expected id=748 | brand=WOW Sports | cat=Sporting Goods > Outdoor Recreation > Boating & Water Sports > Towed Water Sports > Towable Rafts & Tubes
      expected caption: Ace Racing
      top-3 returned:
        id=1434 | brand=Inflatable-Zone | caption=Transpa Inflatable Water Roller, Inflatable Clear Water Rolling Tube
        id=1192 | brand=PieceHouse | caption=Puzzle 